In [14]:
#importar librerias
%pip install -q scikit-image
%pip install -q sklearn.cluster
import numpy as np
import pandas as pd
import os
from skimage import io
import matplotlib.pyplot as plt  
import re

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sklearn.cluster (from versions: none)
ERROR: No matching distribution found for sklearn.cluster


In [15]:
#cargar las imagenes de la carpeta de Frutas
dirname = os.path.join(os.getcwd(), 'train')
imgpath = dirname + os.sep 
 
images = []
directories = []
dircount = []
prevRoot=''
cant=0
 
print("leyendo imagenes de ",imgpath)
 
for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        if re.search("\.(jpg|jpeg|png)$", filename):
            cant=cant+1
            filepath = os.path.join(root, filename)
            image = plt.imread(filepath)
            images.append(image)
            b = "Leyendo..." + str(cant)
            print (b, end="\r")
            if prevRoot !=root:
                print(root, cant)
                prevRoot=root
                directories.append(root)
                dircount.append(cant)
                cant=0
dircount.append(cant)
 
dircount = dircount[1:]
dircount[0]=dircount[0]+1
print('Directorios leidos:',len(directories))
print("Imagenes en cada directorio", dircount)
print('suma Total de imagenes en subdirs:',sum(dircount))
#images[1]
    

<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:15: SyntaxWarning: invalid escape sequence '\.'
C:\Users\carme\AppData\Local\Temp\ipykernel_21204\1728222707.py:15: SyntaxWarning: invalid escape sequence '\.'
  if re.search("\.(jpg|jpeg|png)$", filename):


leyendo imagenes de  c:\Users\carme\source\repos\ImageCategorization\train\
c:\Users\carme\source\repos\ImageCategorization\train\train 1
Directorios leidos: 1
Imagenes en cada directorio [219]
suma Total de imagenes en subdirs: 219


In [16]:
etiquetas=[]
for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        nombre = filename.split("_")[0]   # todo antes del "_"
        etiquetas.append(nombre)

print(etiquetas)

['apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'apple', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'banana', 'ban

In [17]:
#print(type(images))
#print(type(images[0]))
#print(images[0].shape)

In [18]:
#array de imagénes
images_array = np.array(images, dtype=object)

In [19]:
from skimage.transform import resize

def simple_resize(images, new_size=(224, 224)):
    resized = []
    for img in images:
        if img.ndim == 2:
            r = resize(img, new_size, preserve_range=True, anti_aliasing=True)
        else:
            r = resize(img, (new_size[0], new_size[1], img.shape[2]), preserve_range=True, anti_aliasing=True)

        resized.append(r.astype(img.dtype))

    return resized  

In [20]:
resized_images = simple_resize(images_array, new_size=(224 , 224))

### Preprocesado de imágenes ###

In [21]:
# Mostrar las imagenes descargadas
'''
for image in resized_images:
    fig, (ax) = plt.subplots(1)
    fig.set_figwidth(15)
    ax.imshow(image)

'''



'\nfor image in resized_images:\n    fig, (ax) = plt.subplots(1)\n    fig.set_figwidth(15)\n    ax.imshow(image)\n\n'

In [22]:
#Transformar rojo , verde  y azul
imagesRGB =[]
for image in resized_images:
    image_rgb = image[:, :, [0, 1, 2]]  
    imagesRGB.append(image_rgb)

In [23]:
#Mostrar imágenes en RGB
'''
for image in imagesRGB:
    fig, (ax) = plt.subplots(1)
    fig.set_figwidth(15)
    ax.imshow(image)
'''


'\nfor image in imagesRGB:\n    fig, (ax) = plt.subplots(1)\n    fig.set_figwidth(15)\n    ax.imshow(image)\n'

In [24]:
#normalizar imagenes
X = np.array(imagesRGB)
print("Shape:", X.shape)       # debería decir (n_imágenes, 244, 244, 3)
print("Tipo:", X.dtype)        # probablemente uint8
print("Rango original:", X.min(), "a", X.max())
X = X.astype('float32') / 255.0 



Shape: (219, 224, 224, 3)
Tipo: float32
Rango original: 0.0 a 255.0


In [25]:
import numpy as np
import matplotlib.pyplot as plt

print("----- COMPROBACIÓN DE LA NORMALIZACIÓN ----")
print(f"Shape: {X.shape}")
print(f"Tipo de dato: {X.dtype}")
print(f"Valor mínimo: {X.min():.5f}")
print(f"Valor máximo: {X.max():.5f}")

----- COMPROBACIÓN DE LA NORMALIZACIÓN ----
Shape: (219, 224, 224, 3)
Tipo de dato: float32
Valor mínimo: 0.00000
Valor máximo: 1.00000


In [26]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

encoder = LabelEncoder()
Y = encoder.fit_transform(etiquetas)

In [27]:
import pickle

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)


In [28]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_clases = len(np.unique(Y))

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_clases, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#model.summary()


In [29]:
history = model.fit(
    X, Y,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)


Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 409ms/step - accuracy: 0.3756 - loss: 3.2248 - val_accuracy: 0.0000e+00 - val_loss: 1.0406
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 345ms/step - accuracy: 0.5787 - loss: 0.9105 - val_accuracy: 0.0909 - val_loss: 1.2788
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 409ms/step - accuracy: 0.7056 - loss: 0.6758 - val_accuracy: 0.3636 - val_loss: 1.1047
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 459ms/step - accuracy: 0.7513 - loss: 0.5137 - val_accuracy: 0.8636 - val_loss: 0.4975
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 435ms/step - accuracy: 0.8223 - loss: 0.4402 - val_accuracy: 0.5000 - val_loss: 1.3022
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 434ms/step - accuracy: 0.8223 - loss: 0.3787 - val_accuracy: 0.9545 - val_loss: 0.2566
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 442ms/step - accuracy: 0.8477 - loss: 0.3280 - val_accuracy: 0.6364 - val_loss: 0.9806
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 439ms/step - accuracy: 0.8832 - loss: 0.2659 - val_accuracy: 0.9091 - val_l

In [30]:
model.save("modelo_frutas.h5")

In [31]:
import numpy as np
import tensorflow as tf
from PIL import Image
import pickle

model = tf.keras.models.load_model("modelo_frutas.h5")
encoder = pickle.load(open("label_encoder.pkl", "rb"))

def predecir(ruta):
    img = Image.open(ruta).convert("RGB").resize((224, 224))
    img = np.array(img) / 255.0
    img = np.expand_dims(img, axis=0)  

    pred = model.predict(img)
    clase = np.argmax(pred)
    etiqueta = encoder.inverse_transform([clase])[0]

    print("Predicción:", etiqueta)



In [32]:
dirname = os.path.join(os.getcwd(),"image.png")
imgpath = dirname 

predecir(imgpath)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
Predicción: banana


In [33]:
dirname = os.path.join(os.getcwd(),"image2.jpg")
imgpath = dirname 

predecir(imgpath)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicción: orange


Generación de la parte de API

In [ ]:
%pip install fastapi uvicorn pillow
%pip install python-multipart
